# PhishTriage: Colab notebook

Canonical development notebook for an academic phishing-email triage project. It runs top to bottom in
Google Colab on **CPU only**. There is no local backend, Docker or database server.

| Sections | Phase | Content |
|---|---|---|
| 0–6 | 1 | Verified data acquisition, audit, leakage-resistant partitions |
| 7–15 | 2 | Baseline classifiers, artifact-sensitivity analysis, held-out evaluation, reusable inference |

Later phases (evidence tools, playbook retrieval, investigator/reviewer agents, interface) are tracked in
`COLAB_STATUS.md` and are **not implemented yet**.

Run with *Runtime → Run all*. Cells are idempotent, so re-running reuses verified downloads.
The reusable code lives in the repository's `phishtriage/` package; section 0 makes it importable.

## 0. Environment and project code

* Installs only packages that are missing. Colab normally has all of them already.
* Fixes the seed and chooses the working directory: `/content/phishtriage` in Colab, otherwise
  `$PHISHTRIAGE_WORKDIR` or `~/.cache/phishtriage`. Raw data, models and full-text audit files stay there,
  **outside Git**.
* Finds the `phishtriage` package. Locally it uses the repository this notebook lives in. In Colab it clones the
  public repository's `phishtriage-colab` branch into `/content/PhishTriage`; an existing clone is reused as-is.
  The commit used is printed and recorded in the evidence files.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib",
            "requests": "requests", "sklearn": "scikit-learn", "joblib": "joblib", "pyarrow": "pyarrow"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import json, os, platform, random, re, shutil, time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy
import sklearn

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ENVIRONMENT = "colab" if IN_COLAB else "local"

REPO_URL = "https://github.com/nishtrip14/PhishTriage.git"
REPO_REF = "phishtriage-colab"


def find_repo():
    candidates = [Path(os.environ["PHISHTRIAGE_REPO"])] if os.environ.get("PHISHTRIAGE_REPO") else []
    candidates += [Path.cwd(), Path.cwd().parent]
    return next((c.resolve() for c in candidates if (c / "phishtriage" / "__init__.py").exists()), None)


REPO_DIR = find_repo()
if REPO_DIR is None:
    if not IN_COLAB:
        raise RuntimeError("Run this notebook from the repository, or set PHISHTRIAGE_REPO to its path.")
    REPO_DIR = Path("/content/PhishTriage")
    if not (REPO_DIR / "phishtriage" / "__init__.py").exists():
        subprocess.check_call(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def git(*args):
    r = subprocess.run(["git", "-C", str(REPO_DIR), *args], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None


CODE_COMMIT = git("rev-parse", "HEAD")
CODE_DIRTY = bool(git("status", "--porcelain", "--untracked-files=no", "--", "phishtriage", "reports/split_manifest.csv"))

import phishtriage
from phishtriage import data as D, text as T, models as M, inference as I

SEED = D.SEED
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:
    WORKDIR = Path("/content/phishtriage")
else:
    WORKDIR = Path(os.environ.get("PHISHTRIAGE_WORKDIR", Path.home() / ".cache" / "phishtriage"))
RAW_DIR, MANUAL_DIR = WORKDIR / "raw", WORKDIR / "manual"
OUT_DIR, FIG_DIR = WORKDIR / "outputs", WORKDIR / "outputs" / "figures"
AUDIT_DIR, MODEL_DIR = WORKDIR / "audit", WORKDIR / "models"
for d in (RAW_DIR, MANUAL_DIR, FIG_DIR, AUDIT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
# Optional: also copy the small evidence files here (used locally to refresh the repo's reports/).
EXPORT_DIR = Path(os.environ["PHISHTRIAGE_EXPORT_DIR"]) if os.environ.get("PHISHTRIAGE_EXPORT_DIR") else None


def display_path(p: Path) -> str:
    """Show paths without exposing the local home directory."""
    try:
        return "~/" + str(Path(p).relative_to(Path.home()))
    except ValueError:
        return str(p)


print(f"Python {platform.python_version()} | Colab: {IN_COLAB} | CPU-only | {os.cpu_count()} CPUs")
print(" | ".join(f"{m.__name__} {m.__version__}" for m in (np, pd, scipy, sklearn, matplotlib, requests)))
print(f"phishtriage {phishtriage.__version__} | code commit {CODE_COMMIT} {'(modified)' if CODE_DIRTY else ''}")
print("Working directory:", display_path(WORKDIR), "| seed:", SEED)

## 1. Dataset choice and provenance

**First candidate, rejected: MeAJOR v2.0** ([zenodo.org/records/18471483](https://zenodo.org/records/18471483), CC-BY-4.0).
The record says it merges TREC-05/06/07, the Nazario phishing corpus and Nigerian fraud. When inspected on
2026-10-15, its only data file (108,685 rows) contained just the sources `trec5`, `trec6` and `trec7`.
Those are **spam/ham** corpora, so its `label=1` means spam, not phishing.

**Selected:** *Phishing Email Curated Datasets* by A. I. Champa
([zenodo.org/records/8339691](https://zenodo.org/records/8339691), DOI 10.5281/zenodo.8339691, CC-BY-4.0,
open access, no credentials needed). Each source corpus is a separate CSV, so the labels can be traced:

| Source file | Original label | Mapped label | Rationale |
|---|---|---|---|
| `Nazario.csv` | 1 (all rows) | **phishing** | Jose Nazario's phishing corpus, collected from a honeypot mailbox |
| `SpamAssasin.csv` | 0 | **legitimate** | SpamAssassin public corpus *ham* |
| `SpamAssasin.csv` | 1 | *excluded* | Generic spam, not phishing |

Known limitations, which are measured below: source and label are perfectly confounded; there is a large date
gap between the classes; and the curated `urls` column is only a 0/1 flag, with link targets removed from the
phishing bodies. Pinned URLs and checksums are in `phishtriage/data.py`.

## 2. Download with verification

`D.acquire()` reuses a file only if its SHA-256 matches. Otherwise it downloads with timeouts and bounded
retries (client errors such as 404 are not retried), rejects oversized or truncated responses, and checks the
SHA-256 before keeping the file.

If the download fails: in Colab, a file picker opens; locally, place the file in `manual/`. Uploaded files go
through the same schema validation, and a checksum mismatch is recorded, never hidden.

In [ ]:
def colab_upload(spec: dict, manual_dir: Path) -> Path:
    from google.colab import files
    print(f"Please upload {spec['file']} (download it from {D.ZENODO_RECORD}).")
    uploaded = files.upload()
    if spec["file"] not in uploaded:
        raise D.DownloadError(f"Expected an upload named {spec['file']}, got {sorted(uploaded)}")
    dest = manual_dir / spec["file"]
    dest.write_bytes(uploaded[spec["file"]])
    return dest


raw, provenance = {}, {}
for name, spec in D.SOURCES.items():
    path, info = D.acquire(name, spec, RAW_DIR, MANUAL_DIR, upload_fn=colab_upload if IN_COLAB else None)
    df = D.load_and_validate(name, path)
    raw[name] = df
    provenance[name] = {
        **{k: spec[k] for k in ("file", "url", "size", "md5")},
        "expected_sha256": spec["sha256"], "observed_sha256": D.sha256_of(path), **info,
        "rows": len(df), "columns": list(df.columns),
        "checked_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    print(f"{name:13s} {info['status']:28s} rows={len(df):,} columns={list(df.columns)}")

if any(p["observed_sha256"] != p["expected_sha256"] for p in provenance.values()):
    print("WARNING: at least one file does not match the pinned checksum. Its results are not comparable to the "
          "documented dataset version.")

## 3. Normalization

The notebook builds one documented dataframe (`D.normalize_sources`). Missing headers, senders, dates and URLs
are **flagged, never invented**.

| Column | Meaning |
|---|---|
| `record_id` | `<source>:<row index in the pinned file>`, a non-sensitive identifier |
| `source` | `nazario` or `spamassassin` (audit only, **not** a model feature) |
| `subject`, `body` | original text, with missing values as empty strings |
| `text` | `subject + blank line + body`, the **only** model input |
| `original_label` / `mapped_label` | label as given in the source file / `phishing` or `legitimate` |
| `exclusion_reason` | why a row is excluded (empty if included) |
| `missing_*` | whether the source lacks or blanks that field |
| `src_url_flag` | the curator's 0/1 "contains URLs" flag; the URLs themselves are not in the data |
| `year` | year parsed from the `date` header (audit only) |

In [ ]:
data = D.normalize_sources(raw)
print(f"Loaded rows: {len(data):,}")
print("Normalized columns:", list(data.columns))

## 4. Audit

Every number below is computed from the loaded files.

Duplicates are found on *normalized* text: lower-cased, with whitespace collapsed. When identical texts carry
different source/label pairs, **all** copies are excluded as ambiguous. Among the remaining exact duplicates, one
copy is kept (the lowest `record_id`).

In [ ]:
data, audit = D.audit_and_exclude(data)
print(json.dumps({k: audit[k] for k in audit if k != "missing_fields_by_source"}, indent=2))
print("\nMissing or blank fields per source:")
display(pd.DataFrame(audit["missing_fields_by_source"]).T)

### Confounds a classifier could exploit

Each source has a different collection context, so some text patterns identify the **source** rather than
phishing behaviour. This cell measures a fixed list of such artifacts per class, plus the date ranges.

In [ ]:
ARTIFACTS = {
    "collector mailbox (monkey.org)": r"monkey\.org",
    "collector name ('jose')": r"\bjose\b",
    "mailing-list footer": r"listinfo|unsubscribe",
    "SpamAssassin / SourceForge mention": r"spamassassin|sourceforge",
    "bracketed list tag in subject": r"^\s*(?:re:\s*)?\[[^\]]+\]",
    "literal http(s) link": r"https?://",
}
inc = data[data["included"]]
lower = inc["text"].str.lower()
artifact_rates = pd.DataFrame({
    label: {k: round(float(lower[inc["mapped_label"] == label].str.contains(p, regex=True).mean()), 3)
            for k, p in ARTIFACTS.items()}
    for label in ("legitimate", "phishing")})
audit["artifact_prevalence_by_class"] = artifact_rates.to_dict()

years = inc.groupby("mapped_label")["year"].describe(percentiles=[0.05, 0.5, 0.95])[["count", "min", "5%", "50%", "95%", "max"]]
audit["year_by_class"] = {k: {c: (None if pd.isna(v) else float(v)) for c, v in row.items()} for k, row in years.iterrows()}
audit["src_url_flag_rate_by_class"] = inc.groupby("mapped_label")["src_url_flag"].mean().round(3).to_dict()

print("Share of included texts containing each artifact:")
display(artifact_rates)
print("Year parsed from the date header:")
display(years)

### Sanitized examples

Short excerpts with email addresses, URLs, IP addresses, phone-like numbers, reply attributions and From/To/Cc
names redacted (`T.sanitize`). The redaction is heuristic, so review it before publishing any executed copy.

In [ ]:
examples = pd.concat([g.sample(2, random_state=SEED) for _, g in inc.groupby("mapped_label")])
examples = examples[["record_id", "mapped_label", "subject", "body"]].copy()
examples["subject"] = examples["subject"].map(lambda s: T.sanitize(s, 70))
examples["body"] = examples["body"].map(T.sanitize)
with pd.option_context("display.max_colwidth", 170):
    display(examples.reset_index(drop=True))

### Class balance and text length

In [ ]:
# Palette slots 1 and 2 (blue = legitimate, orange = phishing); recessive axes, direct labels.
CLASS_COLORS = {"legitimate": "#2a78d6", "phishing": "#eb6834"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
                     "axes.titlecolor": INK, "axes.titlesize": 11, "font.size": 9})

counts = inc["mapped_label"].value_counts().reindex(["legitimate", "phishing"])
excl = data.loc[~data["included"], "exclusion_reason"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 1.3]})
for ax, series, colors, title, pad in (
        (axes[0], counts, [CLASS_COLORS[c] for c in counts.index], "Included records by class", 1.2),
        (axes[1], excl, INK_2, "Excluded records by reason", 1.25)):
    ax.barh(series.index, series.values, color=colors, height=0.5)
    for i, v in enumerate(series.values):
        ax.text(v, i, f" {v:,}", va="center", color=INK)
    ax.set_title(title); ax.set_xlabel("records"); ax.invert_yaxis()
    ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.set_xlim(0, series.max() * pad)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_balance.png", dpi=120)
plt.show()

lengths = inc.assign(chars=inc["text"].str.len())
bins = np.logspace(1, np.log10(max(lengths["chars"].max(), 100)), 50)
fig, ax = plt.subplots(figsize=(8, 3.2))
for label in ("legitimate", "phishing"):
    vals = lengths.loc[lengths["mapped_label"] == label, "chars"]
    ax.hist(vals, bins=bins, histtype="step", lw=2, color=CLASS_COLORS[label],
            label=f"{label} (median {int(vals.median()):,} chars)")
ax.set_xscale("log"); ax.set_xlabel("combined subject + body length (characters, log scale)"); ax.set_ylabel("records")
ax.set_title("Text length by class"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK)
fig.tight_layout()
fig.savefig(FIG_DIR / "text_length.png", dpi=120)
plt.show()

audit["text_length_chars_by_class"] = {
    k: {s: float(v) for s, v in g.describe(percentiles=[0.05, 0.5, 0.95])[["min", "5%", "50%", "95%", "max"]].items()}
    for k, g in lengths.groupby("mapped_label")["chars"]}

## 5. Leakage-resistant partitions

**Duplicate grouping.** Exact duplicates were already collapsed. Near-duplicates are linked when the **exact
Jaccard similarity** of their word 5-gram shingle sets (CRC32-hashed, deterministic) is ≥ 0.8. Connected
components become `group_id`s. This catches template variants. It does not catch paraphrases.

**Split.** Whole groups go to train/dev/test (about 70/15/15), stratified by group majority label, with a seeded
shuffle. The regenerated split must equal the committed `reports/split_manifest.csv`. A mismatch stops the
notebook, so it can be investigated rather than silently overwritten.

**What grouping does not do:** it prevents duplicate overlap between splits. It does **not** remove the
source–label confound, because every split has Nazario-only phishing and SpamAssassin-only legitimate mail.

In [ ]:
model_df = data[data["included"]].sort_values("record_id").reset_index(drop=True)
t0 = time.time()
comp, n_pairs = D.near_duplicate_groups(model_df["text"].tolist())
model_df["group_id"] = [f"g{c:05d}" for c in comp]
group_sizes = model_df["group_id"].value_counts()
audit["near_duplicates"] = {
    "method": f"exact Jaccard on word {D.SHINGLE_WORDS}-gram shingles (CRC32), threshold {D.NEAR_DUP_JACCARD}",
    "linked_pairs": int(n_pairs),
    "groups_total": int(group_sizes.size),
    "multi_record_groups": int((group_sizes > 1).sum()),
    "records_in_multi_record_groups": int(group_sizes[group_sizes > 1].sum()),
    "largest_group": int(group_sizes.max()),
    "mixed_label_groups": int(model_df.groupby("group_id")["label"].nunique().gt(1).sum()),
}
print(f"Near-duplicate grouping took {time.time() - t0:.1f}s")
print(json.dumps(audit["near_duplicates"], indent=2))

model_df["split"] = D.grouped_stratified_split(model_df)
per_class = D.check_partitions(model_df)
assert D.grouped_stratified_split(model_df).equals(model_df["split"]), "split is not reproducible"

MANIFEST_COLUMNS = ["record_id", "group_id", "split", "mapped_label"]
committed_path = REPO_DIR / "reports" / "split_manifest.csv"
committed = pd.read_csv(committed_path, dtype=str)
regenerated = model_df[MANIFEST_COLUMNS].astype(str)
if not committed.equals(regenerated):
    diff = committed.merge(regenerated, on="record_id", how="outer", suffixes=("_committed", "_now"), indicator=True)
    raise RuntimeError(f"Regenerated split differs from {committed_path.name} "
                       f"({(diff['_merge'] != 'both').sum()} unmatched records). Check data checksums and library "
                       "versions before changing the committed manifest.")
SPLIT_MANIFEST_SHA256 = D.sha256_of(committed_path)
print("All partition assertions passed; regenerated split matches the committed manifest",
      f"(sha256 {SPLIT_MANIFEST_SHA256[:12]}…).")

shares = model_df["split"].value_counts(normalize=True)
audit["splits"] = {s: {"records": int((model_df["split"] == s).sum()), "share": round(float(shares[s]), 4),
                       "groups": int(model_df.loc[model_df["split"] == s, "group_id"].nunique()),
                       **{k: int(v) for k, v in per_class.loc[s].items()}} for s in D.SPLIT_FRACTIONS}
display(per_class.assign(total=per_class.sum(axis=1), share=shares.reindex(per_class.index).round(3)))

## 6. Phase 1 evidence

Small, text-free files are written to `outputs/`: `dataset_provenance.json`, `audit_summary.json`,
`split_manifest.csv` and the figures. Raw data stays in `raw/`. The files are copied into the repository's
`reports/` only when `PHISHTRIAGE_EXPORT_DIR` is set (section 15).

In [ ]:
provenance_doc = {
    "dataset": "Phishing Email Curated Datasets (Champa)",
    "record": D.ZENODO_RECORD, "doi": D.ZENODO_DOI, "license": D.DATASET_LICENSE, "citation": D.CITATION,
    "files": provenance,
    "label_mapping": [{"source": s, "original_label": l, "mapped_label": m,
                       "exclusion_reason": D.EXCLUSION_REASON.get((s, l), "")} for (s, l), m in D.LABEL_MAP.items()],
    "rejected_candidates": [{
        "record": "https://zenodo.org/records/18471483", "name": "MeAJOR v2.0",
        "reason": "Data file contains only TREC-05/06/07 sources (spam/ham); label 1 is spam, not phishing."}],
    "seed": SEED,
    "environment": ENVIRONMENT,
    "code_commit": CODE_COMMIT, "code_modified": CODE_DIRTY,
    "versions": {m.__name__: m.__version__ for m in (np, pd, scipy, matplotlib, requests)} | {"python": platform.python_version()},
}
(OUT_DIR / "dataset_provenance.json").write_text(json.dumps(provenance_doc, indent=2) + "\n")
(OUT_DIR / "audit_summary.json").write_text(json.dumps(audit, indent=2, default=int) + "\n")
model_df[MANIFEST_COLUMNS].to_csv(OUT_DIR / "split_manifest.csv", index=False)
print("Phase 1 evidence written to", display_path(OUT_DIR))

# Phase 2: baseline classifiers and an honest evaluation

## 7. What this evaluation can and cannot show

The dataset is a **confounded benchmark**: every phishing email comes from Nazario's honeypot (2015–2022), and
every legitimate email from SpamAssassin mailing-list ham (about 2002). A classifier can therefore score well by
recognising *which corpus* a message came from.

* A high test score here **is not** demonstrated real-world phishing-detection accuracy, and it says nothing
  about unseen sources.
* Leave-one-source-out evaluation is **not meaningful**: each source contains only one class.
* The artifact-normalized comparison (section 9) is a **sensitivity analysis**. It shows how much known source
  cues matter. It cannot prove that the confound has been removed.
* **External validation is pending.** One focused search on 2026-10-15 found no independent, licensed corpus with
  both phishing and legitimate text that doesn't reuse Nazario/SpamAssassin. Synthetic examples are used only for
  workflow tests.

**Protocol, fixed before any test evaluation:**
1. Use the committed split manifest. Fit every learned transform (TF-IDF vocabulary, IDF, classifier) on **train** only.
2. Pick hyperparameters by dev average precision, and the decision threshold by maximum dev phishing-F1.
   Then freeze and save the configurations.
3. Evaluate the four frozen configurations (2 models × 2 text variants) **once** on test.
4. Predeclared primary model for later agent tools: **TF-IDF + logistic regression on artifact-normalized text**,
   because it depends least on the known source cues. The choice was made before seeing test results.
5. No calibration: there is no separate calibration partition. Scores are uncalibrated model estimates.

## 8. Input-length policy

Train/dev inspection found a few extreme texts: a 4.6M-character phishing message that is mostly a dumped
spam-filter log, and legitimate messages holding encoded binary data or long digests. **Policy:** score the first
`MAX_INPUT_CHARS` (20,000) characters and always flag the input as truncated. Training, evaluation and `predict()`
apply the same rule through the shared `TextPreprocessor` pipeline step. Nothing is dropped.

In [ ]:
model_df["input_chars"] = model_df["text"].str.len()
model_df["truncated"] = model_df["input_chars"] > T.MAX_INPUT_CHARS
trunc = pd.crosstab(model_df["split"], model_df["mapped_label"], values=model_df["truncated"], aggfunc="sum")
trunc = trunc.reindex(list(D.SPLIT_FRACTIONS)).fillna(0).astype(int)
length_policy = {
    "max_input_chars": T.MAX_INPUT_CHARS,
    "rule": "keep the first max_input_chars characters; flag as truncated; same in training and inference",
    "train_p99_5_chars": float(model_df.loc[model_df["split"] == "train", "input_chars"].quantile(0.995)),
    "truncated_by_split_and_class": {s: row.to_dict() for s, row in trunc.iterrows()},
    "truncated_total": int(model_df["truncated"].sum()),
}
print(f"Train 99.5th percentile: {length_policy['train_p99_5_chars']:,.0f} chars; "
      f"limit {T.MAX_INPUT_CHARS:,}; truncated records: {length_policy['truncated_total']}")
display(trunc)

## 9. Artifact normalization (sensitivity variant)

`T.normalize_artifacts` applies the **same** rules to every text, without seeing its label or source. The rules
come from the documented artifacts and from inspecting frequent lines in **train+dev only**. Test data played
no part in choosing them.

They remove source cues; they **cannot** recover the link targets the curators already stripped from phishing
bodies. Removing visible URLs from both classes stops their *presence* from acting as a corpus cue. Writing style
(quoted `>` replies, signatures, topics) is not addressed and can still identify the source.

Original and normalized texts are saved to `audit/normalized_texts.parquet` in the working directory, outside Git,
for manual auditing.

In [ ]:
for i, rule in enumerate(T.ARTIFACT_RULES, 1):
    print(f"{i}. {rule}")

model_df["text_normalized"] = [T.prepare(t, "artifact_normalized")[0] for t in model_df["text"]]
CHECK = {**ARTIFACTS, "four-digit year 1980-2029": r"\b(?:19[89]\d|20[0-2]\d)\b"}
trdev = model_df[model_df["split"] != "test"]
rows = {}
for name, pattern in CHECK.items():
    for col, when in (("text", "before"), ("text_normalized", "after")):
        hit = trdev[col].str.lower().str.contains(pattern, regex=True)
        for label in ("legitimate", "phishing"):
            rows.setdefault(name, {})[f"{label} {when}"] = round(float(hit[trdev["mapped_label"] == label].mean()), 3)
normalization_effect = pd.DataFrame(rows).T[["legitimate before", "legitimate after", "phishing before", "phishing after"]]
empty_after = (model_df["text_normalized"].str.strip() == "")
empty_tab = pd.crosstab(model_df["split"], model_df["mapped_label"], values=empty_after, aggfunc="sum").reindex(list(D.SPLIT_FRACTIONS)).fillna(0).astype(int)
removed_share = (1 - model_df["text_normalized"].str.len() / model_df["text"].str.len().clip(upper=T.MAX_INPUT_CHARS))
normalization_summary = {
    "rules_version": T.ARTIFACT_RULES_VERSION, "rules": T.ARTIFACT_RULES,
    "artifact_prevalence_train_dev": normalization_effect.to_dict(orient="index"),
    "empty_after_normalization_by_split_and_class": {s: r.to_dict() for s, r in empty_tab.iterrows()},
    "median_share_of_characters_removed_train_dev": {
        k: round(float(v), 3) for k, v in removed_share[model_df["split"] != "test"].groupby(model_df["mapped_label"]).median().items()},
}
print("\nShare of train+dev texts containing each artifact, before and after normalization:")
display(normalization_effect)
print("Texts that are empty after normalization (they still receive a score in batch evaluation; predict() abstains):")
display(empty_tab)

model_df[["record_id", "split", "mapped_label", "text", "text_normalized"]].to_parquet(
    AUDIT_DIR / "normalized_texts.parquet", index=False)
print("Saved original vs normalized texts to", display_path(AUDIT_DIR / "normalized_texts.parquet"), "(outside Git)")

## 10. Train on train, select on dev, then freeze

Both baselines use sklearn pipelines: `TextPreprocessor → TfidfVectorizer → classifier`. The vectorizer uses
word 1–2-grams, `min_df=2`, `max_df=0.95`, at most 50,000 features, sublinear TF and sparse float32 matrices.

* **A. Logistic regression:** `class_weight="balanced"`, liblinear solver, C ∈ {0.1, 1, 10}.
* **B. Multinomial NB:** α ∈ {0.01, 0.1, 1}, with learned or uniform class priors.

Each grid point is fitted on train and scored on dev. The best by dev average precision is kept, and its
threshold is the one that maximizes dev phishing-F1. The frozen configurations are saved, with a hash, **before**
the test split is touched.

In [ ]:
train = model_df[model_df["split"] == "train"]
dev = model_df[model_df["split"] == "dev"]
X_train, y_train = train["text"].tolist(), train["label"].astype(int).to_numpy()
X_dev, y_dev = dev["text"].tolist(), dev["label"].astype(int).to_numpy()

frozen = {}
t0 = time.time()
for variant in T.VARIANTS:
    for kind in M.MODEL_KINDS:
        cfg = M.tune_on_dev(kind, variant, X_train, y_train, X_dev, y_dev)
        frozen[cfg["model_id"]] = cfg
print(f"Dev selection took {time.time() - t0:.1f}s")

FROZEN_PATH = OUT_DIR / "frozen_configs.json"
FROZEN_PATH.write_text(json.dumps(frozen, indent=2) + "\n")
FROZEN_SHA256 = D.sha256_of(FROZEN_PATH)
print(f"Frozen configurations saved (sha256 {FROZEN_SHA256[:12]}…). Test split not used yet.")
display(pd.DataFrame([{"model": k, "params": v["params"], "dev AP": round(v["dev_average_precision"], 4),
                       "threshold": round(v["threshold"], 4), "dev F1 @ threshold": round(v["dev_f1_at_threshold"], 4)}
                      for k, v in frozen.items()]).set_index("model"))

## 11. Held-out test evaluation (run once)

Each frozen configuration is refitted on train, with the same seed and data as during selection, and then scored
on test exactly once. Dev metrics at the frozen threshold are shown for reference. Timings are wall-clock times
in **this** runtime.

`predict_proba` gives a model-estimated score. It is **not** a calibrated or validated probability of phishing.

In [ ]:
assert D.sha256_of(FROZEN_PATH) == FROZEN_SHA256, "frozen configurations changed after selection"
test = model_df[model_df["split"] == "test"]
X_test, y_test = test["text"].tolist(), test["label"].astype(int).to_numpy()


def public(metrics: dict) -> dict:
    return {k: v for k, v in metrics.items() if not k.startswith("_")}


fitted, results, test_scores, test_preds = {}, {}, {}, {}
for mid, cfg in frozen.items():
    pipe = M.build_pipeline(cfg["kind"], cfg["variant"], cfg["params"])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    dev_m = M.evaluate(pipe, cfg["threshold"], X_dev, y_dev)
    test_m = M.evaluate(pipe, cfg["threshold"], X_test, y_test)
    fitted[mid], test_scores[mid], test_preds[mid] = pipe, test_m["_scores"], test_m["_y_pred"]
    results[mid] = {"train_records": len(y_train), "fit_seconds": round(fit_s, 3),
                    "vocabulary_size": len(pipe.named_steps["tfidf"].vocabulary_),
                    "dev": public(dev_m), "test": public(test_m)}

comparison = pd.DataFrame([{
    "model": mid, "variant": frozen[mid]["variant"],
    "precision": r["test"]["precision_phishing"], "recall": r["test"]["recall_phishing"],
    "F1": r["test"]["f1_phishing"], "macro F1": r["test"]["macro_f1"], "FPR": r["test"]["false_positive_rate"],
    "avg precision": r["test"]["average_precision"], "FP": r["test"]["confusion_matrix"]["fp"],
    "FN": r["test"]["confusion_matrix"]["fn"], "dev F1": r["dev"]["f1_phishing"], "fit s": r["fit_seconds"],
    "ms/email": r["test"]["inference_ms_per_email"]} for mid, r in results.items()]).set_index("model")
support = results[next(iter(results))]["test"]["support"]
print(f"Test support: {support} (phishing share {y_test.mean():.3f})")
display(comparison.round(4))

In [ ]:
MODEL_COLORS = {"tfidf_logreg": "#2a78d6", "tfidf_mnb": "#eb6834"}
MODEL_NAMES = {"tfidf_logreg": "TF-IDF + logistic regression", "tfidf_mnb": "TF-IDF + multinomial NB"}
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, variant in zip(axes, T.VARIANTS):
    for kind in M.MODEL_KINDS:
        mid = M.model_id(kind, variant)
        p, r, _ = precision_recall_curve(y_test, test_scores[mid])
        ax.plot(r, p, lw=2, color=MODEL_COLORS[kind],
                label=f"{MODEL_NAMES[kind]} (AP {results[mid]['test']['average_precision']:.3f})")
        m = results[mid]["test"]
        ax.plot(m["recall_phishing"], m["precision_phishing"], "o", ms=8, color=MODEL_COLORS[kind],
                markeredgecolor="white", markeredgewidth=2)
    ax.axhline(y_test.mean(), color=INK_2, lw=1, ls="--")
    ax.text(0.01, y_test.mean() + 0.015, "prevalence", color=INK_2)
    ax.set_title(f"{variant.replace('_', '-')} text"); ax.set_xlabel("recall (phishing)")
    ax.set_xlim(0, 1.01); ax.set_ylim(0, 1.02); ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower left", labelcolor=INK)
axes[0].set_ylabel("precision (phishing)")
fig.suptitle("Test precision-recall curves (dots: frozen dev thresholds)", color=INK, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "pr_curves.png", dpi=120)
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, mid in zip(axes, results):
    cm = results[mid]["test"]["confusion_matrix"]
    grid = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
    ax.imshow(grid / grid.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for (i, j), v in np.ndenumerate(grid):
        share = v / grid[i].sum()
        ax.text(j, i, f"{v:,}", ha="center", va="center", color="white" if share > 0.6 else INK)
    ax.set_xticks([0, 1], ["legit", "phish"]); ax.set_yticks([0, 1], ["legit", "phish"])
    ax.set_xlabel("predicted"); ax.set_ylabel("actual" if mid == next(iter(results)) else "")
    kind, variant = mid.split("__")
    ax.set_title(f"{'LR' if kind == 'tfidf_logreg' else 'NB'} · {variant.replace('_', '-')}", fontsize=10)
    for s in ax.spines.values():
        s.set_visible(False)
fig.suptitle("Test confusion matrices (row-normalized shading)", color=INK, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "confusion_matrices.png", dpi=120)
plt.show()

## 12. Influential terms: do source artifacts dominate?

These are the largest positive (phishing) and negative (legitimate) logistic-regression coefficients. They are
learned from the **training** split only. `artifact` marks n-grams that contain a predeclared source-artifact
token (collector name, mail-list and footer vocabulary, URL/domain fragments, years). The token list was fixed
before test evaluation.

In [ ]:
influential = {}
for variant in T.VARIANTS:
    mid = M.model_id("tfidf_logreg", variant)
    influential[mid] = M.top_terms(fitted[mid], k=20)
    side = []
    for direction in ("phishing", "legitimate"):
        tt = influential[mid][direction]
        side.append(pd.DataFrame(tt["terms"]).rename(columns=lambda c: f"{direction} {c}"))
        print(f"{mid}: artifact share among top-20 {direction} terms = {tt['artifact_share']:.0%}")
    with pd.option_context("display.max_rows", 25):
        display(pd.concat(side, axis=1))

## 13. Error examples (sanitized)

Up to three false positives and three false negatives for each logistic-regression variant. The excerpts are
redacted (`T.sanitize`) and truncated.

In [ ]:
error_counts = {}
rows = []
for variant in T.VARIANTS:
    mid = M.model_id("tfidf_logreg", variant)
    pred = test_preds[mid]
    fp_idx, fn_idx = np.where((pred == 1) & (y_test == 0))[0], np.where((pred == 0) & (y_test == 1))[0]
    error_counts[mid] = {"false_positives": int(len(fp_idx)), "false_negatives": int(len(fn_idx)),
                         "errors_truncated_input": int(test["truncated"].to_numpy()[np.r_[fp_idx, fn_idx]].sum())}
    for kind_of_error, idx in (("false positive", fp_idx), ("false negative", fn_idx)):
        for i in idx[:3]:
            r = test.iloc[i]
            rows.append({"model": mid, "error": kind_of_error, "record_id": r["record_id"],
                         "score": round(float(test_scores[mid][i]), 3),
                         "subject": T.sanitize(r["subject"], 60), "excerpt": T.sanitize(r["body"], 140)})
print(json.dumps(error_counts, indent=2))
with pd.option_context("display.max_colwidth", 150):
    display(pd.DataFrame(rows))

## 14. Model artifacts and reusable inference

Each frozen model is saved to `models/<model_id>/` in the working directory, **outside Git**. Alongside it goes a
manifest with package versions, preprocessing version, dataset checksums, split-manifest hash, configuration and
the actual results above.

`I.load_artifact()` checks the model file's SHA-256 against the manifest before unpickling, and it rejects a
different preprocessing version or scikit-learn minor version. A missing artifact raises `ArtifactError`; it never
produces a score.

**Colab:** artifacts disappear with the runtime. Re-running sections 0–14 retrains them in well under a minute
on CPU. To reuse a model, copy the whole `models/<model_id>/` folder (manifest plus model) from a run you
trust, and load it with `I.load_artifact(path)`.

In [ ]:
DATASET_INFO = {"record": D.ZENODO_RECORD, "doi": D.ZENODO_DOI,
                "file_sha256": {n: p["observed_sha256"] for n, p in provenance.items()},
                "split_manifest_sha256": SPLIT_MANIFEST_SHA256, "seed": SEED,
                "frozen_configs_sha256": FROZEN_SHA256, "code_commit": CODE_COMMIT, "code_modified": CODE_DIRTY}
manifests = {}
for mid, pipe in fitted.items():
    path = I.save_artifact(pipe, frozen[mid], MODEL_DIR / mid, dataset=DATASET_INFO, results=results[mid],
                           environment=ENVIRONMENT)
    manifests[mid] = json.loads((path / I.MANIFEST_FILE).read_text())
    print(f"saved {mid}: {(path / I.MODEL_FILE).stat().st_size / 1e6:.1f} MB")

PRIMARY_MODEL = "tfidf_logreg__artifact_normalized"   # predeclared in section 7
model, manifest = I.load_artifact(MODEL_DIR / PRIMARY_MODEL)
reloaded = M.phishing_scores(model, X_test)
assert np.allclose(reloaded, test_scores[PRIMARY_MODEL]), "reloaded model disagrees with the in-memory model"
single = I.predict(model, manifest, text=X_test[0])
assert np.isclose(single["score"], test_scores[PRIMARY_MODEL][0]), "predict() disagrees with batch scoring"
print("Reloaded artifact reproduces in-memory test scores; predict() matches batch scoring.")

demo = {
    "test email (shown without text)": I.predict(model, manifest, subject=test.iloc[0]["subject"], body=test.iloc[0]["body"]),
    "empty input": I.predict(model, manifest, subject="", body="   "),
    "oversized input": I.predict(model, manifest, subject="Account notice",
                                 body="Please verify your account details. " * 1500),
}
for name, out in demo.items():
    print(f"\n--- {name}")
    print(json.dumps({k: v for k, v in out.items() if k != "limitations"}, indent=2))
print("\nLimitations attached to every prediction:")
for item in I.LIMITATIONS:
    print(" -", item)

## 15. Phase 2 evidence

These text-free files are added to `outputs/`: `phase2_metrics.json`, `phase2_model_comparison.csv`,
`model_manifests.json`, `frozen_configs.json`, and the PR-curve and confusion-matrix figures. When
`PHISHTRIAGE_EXPORT_DIR` is set, all phase 1 and phase 2 evidence is copied there.
Model binaries and texts are never exported.

In [ ]:
phase2 = {
    "environment": ENVIRONMENT, "cpu_count": os.cpu_count(), "platform": platform.platform(terse=True),
    "versions": I.environment_versions(), "code_commit": CODE_COMMIT, "code_modified": CODE_DIRTY,
    "run_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "dataset": DATASET_INFO, "protocol": {
        "fit_on": "train", "selection": M.SELECTION_METRIC, "threshold": M.THRESHOLD_RULE,
        "test_evaluations_per_config": 1, "calibration": "none (no separate calibration partition)",
        "primary_model_predeclared": PRIMARY_MODEL, "tfidf": {k: str(v) for k, v in M.TFIDF_PARAMS.items()},
        "grids": M.PARAM_GRIDS},
    "length_policy": length_policy, "artifact_normalization": normalization_summary,
    "frozen_configs": frozen, "results": results, "influential_terms": influential, "error_counts": error_counts,
    "external_validation": "pending: no independent licensed phishing+legitimate corpus found on 2026-10-15",
}
(OUT_DIR / "phase2_metrics.json").write_text(json.dumps(phase2, indent=2) + "\n")
comparison.round(4).to_csv(OUT_DIR / "phase2_model_comparison.csv")
(OUT_DIR / "model_manifests.json").write_text(json.dumps(manifests, indent=2) + "\n")

EVIDENCE = ["dataset_provenance.json", "audit_summary.json", "split_manifest.csv", "phase2_metrics.json",
            "phase2_model_comparison.csv", "model_manifests.json", "frozen_configs.json",
            "figures/class_balance.png", "figures/text_length.png", "figures/pr_curves.png",
            "figures/confusion_matrices.png"]
if EXPORT_DIR:
    (EXPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    for p in EVIDENCE:
        shutil.copy2(OUT_DIR / p, EXPORT_DIR / p)
    print("Exported evidence to", display_path(EXPORT_DIR))
for p in EVIDENCE:
    print(f"{p:32s} {(OUT_DIR / p).stat().st_size:>9,} bytes")

print("\nPHASE 2 COMPLETE (environment: %s, code %s%s)" % (ENVIRONMENT, CODE_COMMIT, " modified" if CODE_DIRTY else ""))
print(f"  data: loaded={audit['rows_loaded']:,} included={int(data['included'].sum()):,} | "
      + " ".join(f"{s}={v['records']:,}" for s, v in audit["splits"].items()))
for mid, r in results.items():
    t = r["test"]
    print(f"  {mid:36s} test F1={t['f1_phishing']:.3f} AP={t['average_precision']:.3f} FPR={t['false_positive_rate']:.3f}")